# 01. Foundations: 2026년 7월 VLM 논문 지도 만들기

## 목표

이 노트북은 최신 VLM 논문 5개를 단순 요약이 아니라 학습 가능한 구조로 정리하는 연습입니다.

- 논문을 카테고리, 문제의식, 핵심 기법으로 분해합니다.
- OCR 텍스트와 원본 문서 이미지 표현의 차이를 toy data로 확인합니다.
- VLM 트렌드를 읽을 때 반복해서 사용할 수 있는 질문 목록을 만듭니다.

## 실행 방법

Python 표준 라이브러리만 사용합니다. Jupyter에서 위에서 아래로 실행하면 됩니다.


In [ ]:
# 최신 논문을 읽을 때는 제목만 보지 말고, 문제-기법-출력-평가 축으로 나눠 봅니다.
# 리스트 안의 dict는 작은 데이터베이스처럼 사용할 수 있는 가장 단순한 구조입니다.
papers = [
    {
        "rank": 1,
        "title": "Scalable Visual Pretraining for Language Intelligence",
        "arxiv": "2607.09657",
        "category": "native_visual_pretraining",
        "problem": "텍스트 추출이 문서의 시각 구조를 잃어버린다",
        "method": "원본 문서 이미지 기반 비주얼 사전학습",
        "output": "시각 표현 또는 언어 지능 벤치마크 성능",
    },
    {
        "rank": 2,
        "title": "Unlimited OCR Works",
        "arxiv": "2606.23050",
        "category": "memory_efficient_long_ocr",
        "problem": "긴 OCR 출력에서 KV 캐시가 계속 커진다",
        "method": "Reference Sliding Window Attention",
        "output": "장문 문서 전사 텍스트",
    },
    {
        "rank": 3,
        "title": "Vision as Unified Multimodal Generation",
        "arxiv": "2607.06560",
        "category": "unified_multimodal_generation",
        "problem": "비전 태스크마다 별도 헤드가 필요하다",
        "method": "텍스트/이미지 생성 공간으로 출력 통합",
        "output": "텍스트, 이미지, 혼합 응답",
    },
    {
        "rank": 4,
        "title": "Vision Pretraining for Dense Spatial Perception",
        "arxiv": "2607.05247",
        "category": "dense_spatial_perception",
        "problem": "의미 중심 표현이 경계와 깊이 정보를 놓친다",
        "method": "Masked Boundary Modeling",
        "output": "깊이, 경계, 공간 표현",
    },
    {
        "rank": 5,
        "title": "Infinite Worlds with Versatile Interactions",
        "arxiv": "2607.07534",
        "category": "interactive_world_model",
        "problem": "월드 모델이 긴 상호작용과 실시간성을 함께 달성하기 어렵다",
        "method": "실시간 variant와 agentic harness",
        "output": "행동 조건부 비디오/월드 상태",
    },
]

def print_table(rows, columns):
    """간단한 Markdown 표를 출력합니다.
    
    pandas 없이도 표를 볼 수 있게 직접 구현했습니다.
    학습 자료에서는 의존성을 줄이면 실행 실패 가능성이 낮아집니다.
    """
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    print(header)
    print(sep)
    for row in rows:
        print("| " + " | ".join(str(row[col]) for col in columns) + " |")

print_table(papers, ["rank", "category", "arxiv", "title"])


## 1. 카테고리 태그로 읽기

논문 목록을 처음 볼 때는 모델 이름보다 카테고리를 먼저 잡는 것이 좋습니다. 같은 카테고리 안에서 이전 방법과 무엇이 달라졌는지 비교하기 쉬워집니다.


In [ ]:
# 카테고리를 상위 트렌드로 묶습니다.
# 여러 논문이 같은 문제 축을 공유할 수 있으므로 1개 논문에 1개 이상의 trend tag를 줄 수도 있습니다.
trend_tags = {
    "native_visual_pretraining": ["document_ai", "pretraining", "layout_reasoning"],
    "memory_efficient_long_ocr": ["document_ai", "long_context", "inference_efficiency"],
    "unified_multimodal_generation": ["generalist_model", "output_serialization", "instruction_tuning"],
    "dense_spatial_perception": ["spatial_intelligence", "self_supervised_learning", "embodied_ai"],
    "interactive_world_model": ["world_model", "agent_loop", "real_time_generation"],
}

for paper in papers:
    paper["trend_tags"] = trend_tags[paper["category"]]

for paper in papers:
    print(f"#{paper['rank']} {paper['title']}")
    print("  tags:", ", ".join(paper["trend_tags"]))
    print("  problem:", paper["problem"])
    print()


## 2. 텍스트 추출과 원본 시각 구조의 차이

문서 OCR을 하면 글자는 남지만 위치 정보가 약해질 수 있습니다. 아래 toy document는 제목, 수식, 도표가 있는 페이지를 간단한 격자로 표현합니다.


In [ ]:
# 실제 문서 이미지는 픽셀 배열이지만, 여기서는 이해를 위해 문자열 격자로 표현합니다.
# 각 줄의 위치 자체가 레이아웃 정보입니다.
page_grid = [
    "TITLE: Energy Report             ",
    "                                  ",
    "E = m c^2        [ diagram ]      ",
    "                 [ arrows  ]      ",
    "Table: year | value              ",
    "       2025 | 10                 ",
    "       2026 | 18                 ",
]

# OCR 텍스트는 보통 줄을 이어 붙이거나, 위치 좌표를 약하게 보존합니다.
# 왜 문제가 되는가: 수식 오른쪽의 diagram이 어떤 줄과 연결되는지 잃기 쉽습니다.
plain_text = " ".join(line.strip() for line in page_grid if line.strip())

print("[원본 격자]")
for line in page_grid:
    print(line)

print("\n[plain text로 뭉갠 결과]")
print(plain_text)


In [ ]:
# 시각 토큰을 흉내 내기 위해 텍스트 조각과 좌표를 함께 저장합니다.
# bbox는 (row, col_start, col_end)입니다. 실제 VLM에서는 이 좌표가 이미지 패치 위치나 2D positional encoding으로 들어갑니다.
visual_tokens = [
    {"text": "TITLE", "bbox": (0, 0, 5)},
    {"text": "Energy Report", "bbox": (0, 7, 20)},
    {"text": "E = m c^2", "bbox": (2, 0, 9)},
    {"text": "diagram", "bbox": (2, 18, 25)},
    {"text": "arrows", "bbox": (3, 20, 26)},
    {"text": "Table", "bbox": (4, 0, 5)},
    {"text": "2025", "bbox": (5, 7, 11)},
    {"text": "10", "bbox": (5, 14, 16)},
    {"text": "2026", "bbox": (6, 7, 11)},
    {"text": "18", "bbox": (6, 14, 16)},
]

def same_row_neighbors(token, tokens, max_distance=12):
    """같은 줄에서 가까운 토큰을 찾습니다.
    
    단순한 함수지만, '문서 이해는 텍스트 내용 + 위치 관계'라는 점을 보여줍니다.
    """
    row, start, end = token["bbox"]
    neighbors = []
    for other in tokens:
        if other is token:
            continue
        other_row, other_start, other_end = other["bbox"]
        distance = min(abs(other_start - end), abs(start - other_end))
        if other_row == row and distance <= max_distance:
            neighbors.append(other["text"])
    return neighbors

for token in visual_tokens:
    print(f"{token['text']:<14} -> neighbors: {same_row_neighbors(token, visual_tokens)}")


## 3. 논문 읽기 질문 템플릿

아래 질문은 어떤 VLM 논문에도 반복해서 적용할 수 있습니다. 논문을 읽고 답을 채우면 요약 품질이 크게 좋아집니다.


In [ ]:
reading_questions = [
    "이 논문이 해결하려는 병목은 입력, 모델 구조, 출력, 학습 데이터, 평가 중 어디에 있는가?",
    "기존 방법이 버리던 정보는 무엇인가? 텍스트, 위치, 경계, 시간, 행동 중 어느 축인가?",
    "새로운 기법은 학습 목표를 바꾸는가, attention을 바꾸는가, 출력 포맷을 바꾸는가?",
    "성능 향상이 모델 크기 때문인지, 데이터 변환 때문인지, 추론 구조 때문인지 분리되어 있는가?",
    "실무 적용 시 실패할 수 있는 입력은 무엇인가? 긴 문서, 복잡한 표, 희귀 언어, 작은 객체, 빠른 상호작용 등인가?",
]

for index, question in enumerate(reading_questions, start=1):
    print(f"{index}. {question}")


## 마무리 체크

이제 각 논문을 다음 네 칸으로 요약해 보세요.

| 논문 | 버려지던 정보 | 새로 보존하는 정보 | 실무 적용 후보 |
|---|---|---|---|
| Scalable VP | OCR 전처리에서 사라지는 레이아웃 | 원본 문서의 2D 구조 | 과학 문서 RAG |
| Unlimited OCR | 긴 출력의 전체 캐시 관리 가능성 | 고정 참조와 최근 출력 창 | 대용량 문서 OCR |
| SenseNova-Vision | 태스크별 출력의 통일성 | 텍스트/이미지 생성 규약 | 범용 비전 API |
| LingBot-Vision | 경계와 깊이 불연속 | 경계 중심 공간 표현 | 로봇/자율주행 인식 |
| LingBot-World 2.0 | 긴 상호작용 상태 | 행동 조건부 월드 업데이트 | 실시간 시뮬레이터 |
